# Kimi K2.5 Attention Scan FLOPs Constant

This notebook converts `kimi25_attention_scan_flops_constant.html` into a block-by-block calculation notebook.

Target result:

`attention_scan_flops_per_pair = 8,495,104`

Scope contract: single giant GPU, no parallelism, multiply-add counted as 2 FLOPs, absorbed MLA scan only. This is not per token by itself. Total attention FLOPs need an attention-pair count.

## 1. What is a context pair?

A context pair means one query token looking at one cached context position, across all 64 heads on the single giant GPU.

Total attention scan FLOPs are:

`attention_pair_count * attention_scan_flops_per_pair`

Cached prefix tokens are not charged full linear work again, but they can still be keys/values in context. That is why cached tokens can still create attention pairs.

## 2. Model constants used by the scan

These names intentionally match the HTML pages and the full-model architecture map.

In [ ]:
L = 61
num_attention_heads = 64
heads = num_attention_heads

kv_lora = 512
qk_rope = 64
qk_nope = 128
v_head = 128

assert heads == 64

{
    'L': L,
    'heads': heads,
    'kv_lora': kv_lora,
    'qk_rope': qk_rope,
    'qk_nope': qk_nope,
    'v_head': v_head,
}

## 3. Absorbed MLA scan shape

Kimi uses absorbed MLA. Instead of materializing full per-head K/V tensors for the cache, the scan works against latent cache state plus a small RoPE key.

For one query-context pair, each head does three scan operations:

- Content QK: `q_content[512] dot c_j[512]`.
- RoPE QK: `q_rope[64] dot k_rope_j[64]`.
- AV latent accumulation: `attention_weight * c_j[512]`.

Here `c_j[512]` means the normalized latent content vector used by the attention scan.

In [ ]:
content_qk_per_layer = heads * 2 * kv_lora
rope_qk_per_layer = heads * 2 * qk_rope
av_latent_per_layer = heads * 2 * kv_lora

attention_scan_flops_per_pair_per_layer = (
    content_qk_per_layer
    + rope_qk_per_layer
    + av_latent_per_layer
)

assert content_qk_per_layer == 65_536
assert rope_qk_per_layer == 8_192
assert av_latent_per_layer == 65_536
assert attention_scan_flops_per_pair_per_layer == 139_264

{
    'content_qk_per_layer': content_qk_per_layer,
    'rope_qk_per_layer': rope_qk_per_layer,
    'av_latent_per_layer': av_latent_per_layer,
    'attention_scan_flops_per_pair_per_layer': attention_scan_flops_per_pair_per_layer,
}

## 4. Across all layers

The pair constant multiplies the per-layer pair cost by `L=61` layers.

In [ ]:
attention_scan_flops_per_pair = (
    L * attention_scan_flops_per_pair_per_layer
)

assert attention_scan_flops_per_pair == 8_495_104

{
    'attention_scan_flops_per_pair_per_layer': attention_scan_flops_per_pair_per_layer,
    'L': L,
    'attention_scan_flops_per_pair': attention_scan_flops_per_pair,
}

## 5. Compact formula sanity check

The compact equivalent formula is:

`L * 2 * heads * (2 * kv_lora + qk_rope)`

In [ ]:
compact = L * 2 * heads * (2 * kv_lora + qk_rope)

assert compact == attention_scan_flops_per_pair
assert compact == 8_495_104

compact

## 6. Why `qk_nope` and `v_head` do not appear in the pair-scan constant

`qk_nope=128` does not appear because the no-position query is multiplied by absorbed `W_KC` before the scan. After absorption, the scan sees a 512-dim latent content query, so the scan cost uses `kv_lora=512`.

`v_head=128` does not appear because the scan accumulates latent `kv_lora=512` values. The later absorbed `W_VC` projection maps latent output to `v_head=128`; that projection is charged in the per-token linear FLOPs notebook.

All 64 heads appear because this is the no-parallelism baseline.

## 7. Optional: compute attention pairs for simple request shapes

This cell is not needed to derive the constant, but it shows how to use the constant.

For a no-cache request with `input_tokens` prompt tokens and `output_tokens` generated tokens:

- Prompt/prefill pairs are triangular: `input_tokens * (input_tokens + 1) // 2`.
- Decode pairs: each output query attends to the full prompt plus previously generated output tokens.
- Total scan FLOPs: `pairs * attention_scan_flops_per_pair`.

In [ ]:
def no_cache_attention_pairs(input_tokens: int, output_tokens: int) -> dict:
    prefill_pairs = input_tokens * (input_tokens + 1) // 2
    decode_pairs = output_tokens * input_tokens + output_tokens * (output_tokens - 1) // 2
    total_pairs = prefill_pairs + decode_pairs
    return {
        'prefill_pairs': prefill_pairs,
        'decode_pairs': decode_pairs,
        'total_pairs': total_pairs,
        'attention_scan_flops': total_pairs * attention_scan_flops_per_pair,
    }

example = no_cache_attention_pairs(input_tokens=10, output_tokens=10)

assert example['prefill_pairs'] == 55
assert example['decode_pairs'] == 145
assert example['total_pairs'] == 200
assert example['attention_scan_flops'] == 200 * 8_495_104

example

## 8. What this constant does not include

Excluded from this constant:

- `qkv_a`, `q_b`, `W_KC`, `W_VC`, and `o_proj`; those are per-forward-token linear work.
- Softmax, scale, mask, exp, and reductions.
- RMSNorm and RoPE elementwise rotation.
- KV cache bytes and memory bandwidth.
- MoE, MLP, and router work.

## 9. Source anchors

- HTML source converted from: `z_local/benchserving/mfu_metrics/kimi25_attention_scan_flops_constant.html`.
- Public config source: `moonshotai/Kimi-K2.5/config.json`.
- Local absorbed MLA source: `python/sglang/srt/models/deepseek_common/attention_forward_methods/forward_mla.py`.
- Local model source: `python/sglang/srt/models/deepseek_v2.py` for `attn_mqa`, `kv_lora_rank + qk_rope_head_dim`, and `v_head_dim=kv_lora_rank`.